# TIL - PHP와 MySQL을 연결한 데이터베이스 웹 애플리케이션 실습

## 오늘의 학습 목표

오늘은 지금까지 학습한 데이터베이스 기본 개념을 실제 웹 애플리케이션과 연결해 보았다.

특히 다음 흐름을 직접 확인하는 것이 핵심이었다.

```text
웹 브라우저
    ↓
PHP
    ↓
SQL
    ↓
MySQL
    ↓
SQL 실행 결과
    ↓
PHP
    ↓
HTML
    ↓
웹 브라우저
```

단순히 DBeaver에서 SQL만 실행하는 것이 아니라, 실제 웹 화면에서 데이터베이스의 데이터를 조회하고 표시하는 과정을 실습했다.

---

# 1. 실습 환경

이번 실습에서 사용한 환경은 다음과 같다.

```text
OS: Windows 11

웹 서버 환경:
XAMPP

사용 구성:
- Apache
- PHP
- MySQL

DB 관리 도구:
- DBeaver

코드 작성:
- VS Code

Database:
- course_system

MySQL 연결 정보:
Host: localhost
Port: 3306
User: root
Password: 1234
```

프로젝트 경로는 다음과 같이 구성했다.

```text
C:\xampp\htdocs\course-admin
```

---

# 2. 데이터베이스 웹 관리 시스템 기획

이번 실습에서는 온라인 강좌 관리 시스템을 만들었다.

기본 페이지는 총 5개로 구성했다.

```text
1. Dashboard
2. Students
3. Instructors
4. Courses
5. Enrollments
```

각 페이지의 역할은 다음과 같다.

## Dashboard

전체 데이터 현황을 한눈에 확인한다.

```text
- 전체 학생 수
- 전체 강사 수
- 전체 강좌 수
- 전체 수강 신청 수
- 최근 수강 신청 10건
```

## Students

학생 정보를 조회한다.

```text
- 학생 ID
- 이름
- 이메일
- 전공
- 등록일
```

추가로 간단한 학생 등록 기능도 구현했다.

## Instructors

강사 정보를 조회한다.

```text
- 강사 ID
- 이름
- 이메일
- 소속
- 등록일
```

## Courses

강좌 정보를 조회한다.

```text
- 강좌 ID
- 강좌명
- 강사명
- 카테고리
- 정원
```

강사의 ID를 그대로 보여주는 것이 아니라 JOIN을 사용하여 강사명을 표시한다.

## Enrollments

수강 신청 정보를 조회한다.

```text
- 수강 ID
- 학생명
- 강좌명
- 강사명
- 수강 상태
- 신청일
```

이 페이지에서는 여러 테이블을 JOIN하는 것이 핵심이다.

---

# 3. 프로젝트 구조 만들기

프로젝트 폴더는 다음과 같이 구성했다.

```text
course-admin/
├── config/
│   └── db.php
├── index.php
├── students.php
├── instructors.php
├── courses.php
├── enrollments.php
├── test_db.php
└── assets/
    ├── css/
    │   └── custom.css
    └── js/
        └── app.js
```

각 파일의 역할은 다음과 같다.

```text
config/db.php
→ MySQL 데이터베이스 연결

index.php
→ Dashboard

students.php
→ 학생 관리

instructors.php
→ 강사 관리

courses.php
→ 강좌 관리

enrollments.php
→ 수강 신청 관리

test_db.php
→ DB 연결 테스트

custom.css
→ CSS 스타일

app.js
→ JavaScript 기능
```

---

# 4. PHP와 MySQL 연결

가장 먼저 `config/db.php` 파일을 만들어 데이터베이스 연결 정보를 분리했다.

```php
<?php

$host = 'localhost';
$port = '3306';
$dbname = 'course_system';
$username = 'root';
$password = '1234';

$dsn = "mysql:host={$host};port={$port};dbname={$dbname};charset=utf8mb4";

$options = [
    PDO::ATTR_ERRMODE => PDO::ERRMODE_EXCEPTION,
    PDO::ATTR_DEFAULT_FETCH_MODE => PDO::FETCH_ASSOC,
    PDO::ATTR_EMULATE_PREPARES => false,
];

try {
    $pdo = new PDO($dsn, $username, $password, $options);
} catch (PDOException $e) {
    exit('DB 연결 실패: ' . $e->getMessage());
}
```

PHP에서는 PDO를 사용하여 MySQL에 연결했다.

```text
PHP
 ↓
PDO
 ↓
MySQL
 ↓
course_system
```

---

# 5. DB 연결 테스트

DB 연결 여부를 확인하기 위해 `test_db.php`를 만들었다.

```php
<?php

require_once __DIR__ . '/config/db.php';

echo "DB 연결 성공!";
```

브라우저에서 다음 주소로 확인했다.

```text
http://localhost/course-admin/test_db.php
```

화면에 다음 문구가 나타나면 성공이다.

```text
DB 연결 성공!
```

---

# 6. 실습 중 발생한 404 오류

처음에는 브라우저에서 다음 오류가 발생했다.

```text
404 Not Found
```

원인은 PHP 파일의 실제 확장자가 잘못되어 있었기 때문이다.

예를 들어 다음과 같이 저장되어 있으면 문제가 발생한다.

```text
test_db.php.txt
```

정상 파일명은 다음과 같다.

```text
test_db.php
```

Windows 탐색기에서 다음 옵션을 활성화하여 실제 확장자를 확인했다.

```text
보기
→ 표시
→ 파일 이름 확장명
```

---

# 7. Dashboard 만들기

`index.php`에서 각 테이블의 전체 데이터 수를 조회했다.

```php
$studentCount = $pdo->query("
    SELECT COUNT(*)
    FROM students
")->fetchColumn();

$instructorCount = $pdo->query("
    SELECT COUNT(*)
    FROM instructors
")->fetchColumn();

$courseCount = $pdo->query("
    SELECT COUNT(*)
    FROM courses
")->fetchColumn();

$enrollmentCount = $pdo->query("
    SELECT COUNT(*)
    FROM enrollments
")->fetchColumn();
```

이를 통해 다음 정보를 Dashboard에 표시할 수 있었다.

```text
학생 수
강사 수
강좌 수
수강 신청 수
```

---

# 8. 최근 수강 신청 10건 조회

최근 수강 신청은 JOIN을 사용하여 학생명과 강좌명을 함께 표시했다.

```sql
SELECT
    e.enrollment_id,
    s.name AS student_name,
    c.title AS course_title,
    e.status,
    e.enrolled_at
FROM enrollments e
JOIN students s
    ON e.student_id = s.student_id
JOIN courses c
    ON e.course_id = c.course_id
ORDER BY e.enrolled_at DESC
LIMIT 10;
```

여기서 알게 된 점은 다음과 같다.

```text
enrollments.student_id
        ↓ JOIN
students.name
```

```text
enrollments.course_id
        ↓ JOIN
courses.title
```

DB에는 숫자 ID가 저장되어 있어도, 웹 사용자에게는 사람이 이해하기 쉬운 이름을 보여주는 것이 좋다.

---

# 9. Students 페이지

학생 목록을 조회했다.

```sql
SELECT
    student_id,
    name,
    email,
    major,
    created_at
FROM students
ORDER BY student_id DESC
LIMIT 50;
```

최신 학생부터 표시하기 위해 다음 조건을 사용했다.

```sql
ORDER BY student_id DESC
```

그리고 너무 많은 데이터를 한 번에 표시하지 않도록 다음 조건도 사용했다.

```sql
LIMIT 50
```

---

# 10. 학생 등록 기능

Students 페이지에 간단한 등록 Form을 만들었다.

입력 항목은 다음과 같다.

```text
이름
이메일
전공
```

전체 흐름은 다음과 같다.

```text
HTML Form
    ↓
POST
    ↓
students.php
    ↓
Prepared Statement
    ↓
INSERT
    ↓
MySQL
    ↓
학생 목록 재조회
```

INSERT SQL은 다음과 같다.

```sql
INSERT INTO students (name, email, major)
VALUES (:name, :email, :major);
```

PHP에서는 Prepared Statement를 사용했다.

```php
$stmt = $pdo->prepare($sql);

$stmt->execute([
    'name' => $name,
    'email' => $email,
    'major' => $major,
]);
```

사용자 입력값을 SQL 문자열에 직접 연결하지 않는 것이 중요하다.

---

# 11. Prepared Statement

잘못된 방식은 사용자 입력값을 SQL 문자열에 바로 붙이는 것이다.

예:

```php
$sql = "SELECT * FROM students WHERE email = '" . $_GET['email'] . "'";
```

이 방식은 SQL Injection 문제를 만들 수 있다.

따라서 다음과 같이 Prepared Statement를 사용한다.

```php
$sql = "
    SELECT *
    FROM students
    WHERE email = :email
";

$stmt = $pdo->prepare($sql);

$stmt->execute([
    'email' => $_GET['email']
]);
```

핵심은 다음과 같다.

```text
SQL 구조
+
사용자 입력값

을 분리해서 DB에 전달한다.
```

---

# 12. Instructors 페이지

강사 목록은 다음 데이터를 조회했다.

```sql
SELECT
    instructor_id,
    name,
    email,
    department,
    created_at
FROM instructors
ORDER BY instructor_id DESC
LIMIT 50;
```

화면에는 다음 정보를 표시했다.

```text
강사 ID
이름
이메일
소속
등록일
```

---

# 13. Courses 페이지와 JOIN

Courses 페이지에서는 강사 ID를 그대로 보여주지 않고 JOIN을 이용해 강사명을 표시했다.

```sql
SELECT
    c.course_id,
    c.title,
    c.category,
    c.capacity,
    i.name AS instructor_name
FROM courses c
JOIN instructors i
    ON c.instructor_id = i.instructor_id
ORDER BY c.course_id DESC
LIMIT 100;
```

핵심 JOIN은 다음 부분이다.

```sql
FROM courses c
JOIN instructors i
    ON c.instructor_id = i.instructor_id
```

즉,

```text
courses.instructor_id
       ↓
JOIN
       ↓
instructors.instructor_id
       ↓
instructors.name
```

이 과정을 통해 강사 ID 대신 강사 이름을 화면에 표시할 수 있었다.

---

# 14. Enrollments 페이지와 다중 JOIN

이번 실습에서 가장 중요한 JOIN이었다.

`enrollments`에는 기본적으로 학생과 강좌의 ID가 들어 있다.

```text
student_id
course_id
```

하지만 웹 화면에서는 다음 정보를 보여주고 싶었다.

```text
학생명
강좌명
강사명
```

이를 위해 총 4개 테이블을 연결했다.

```sql
SELECT
    e.enrollment_id,
    s.name AS student_name,
    c.title AS course_title,
    i.name AS instructor_name,
    e.status,
    e.enrolled_at
FROM enrollments e
JOIN students s
    ON e.student_id = s.student_id
JOIN courses c
    ON e.course_id = c.course_id
JOIN instructors i
    ON c.instructor_id = i.instructor_id
ORDER BY e.enrolled_at DESC
LIMIT 100;
```

테이블 연결 구조는 다음과 같다.

```text
enrollments
    ↓
students
    ↓
courses
    ↓
instructors
```

조금 더 정확하게 표현하면 다음과 같다.

```text
enrollments.student_id
        ↓
students.student_id
        ↓
학생명
```

```text
enrollments.course_id
        ↓
courses.course_id
        ↓
강좌명
```

```text
courses.instructor_id
        ↓
instructors.instructor_id
        ↓
강사명
```

---

# 15. 오늘 사용한 주요 SQL

## SELECT

```sql
SELECT *
FROM students;
```

데이터를 조회한다.

## COUNT

```sql
SELECT COUNT(*)
FROM students;
```

전체 데이터 개수를 구한다.

## ORDER BY

```sql
ORDER BY student_id DESC
```

조회 결과의 순서를 정한다.

## LIMIT

```sql
LIMIT 50
```

가져오는 데이터 개수를 제한한다.

## INSERT

```sql
INSERT INTO students (name, email, major)
VALUES (:name, :email, :major);
```

새로운 데이터를 추가한다.

## JOIN

```sql
JOIN instructors i
    ON c.instructor_id = i.instructor_id
```

서로 관련된 테이블을 연결한다.

---

# 16. 웹과 데이터베이스의 연결 이해

오늘 가장 중요하게 이해한 구조는 다음과 같다.

```text
사용자가 웹 페이지 요청

        ↓

Apache

        ↓

PHP 실행

        ↓

PDO를 통해 MySQL 연결

        ↓

SQL 실행

        ↓

MySQL 결과 반환

        ↓

PHP가 HTML 생성

        ↓

브라우저에 결과 출력
```

브라우저가 MySQL에 직접 접속하는 것이 아니다.

중간에서 PHP가 데이터베이스와 통신하고 결과를 웹 화면으로 만들어준다.

---

# 17. CRUD와 오늘 실습의 관계

CRUD는 데이터베이스 애플리케이션의 기본 기능이다.

```text
C = Create
R = Read
U = Update
D = Delete
```

오늘 실습에서는 특히 다음 기능을 직접 사용했다.

```text
Create
→ 학생 등록
→ INSERT

Read
→ 학생/강사/강좌/수강신청 조회
→ SELECT
```

현재는 CRUD 중 `Create`, `Read`를 웹 애플리케이션에서 직접 연결해본 단계이다.

추후에는 `Update`, `Delete` 기능도 추가할 수 있다.

---

# 18. AI 바이브코딩에서 배운 점

AI에게 단순히 다음처럼 요청하는 것보다

```text
웹사이트 만들어줘
```

데이터 구조와 요구사항을 구체적으로 전달하는 것이 중요하다는 것을 알게 되었다.

예를 들어 다음을 명확하게 정의해야 한다.

```text
어떤 DB를 사용하는가?

어떤 테이블이 있는가?

각 테이블은 어떻게 연결되는가?

어떤 페이지가 필요한가?

페이지에서 어떤 데이터를 보여줄 것인가?

JOIN이 필요한 부분은 어디인가?

사용자 입력은 어떻게 처리할 것인가?
```

AI가 코드를 만들어 주더라도 사람이 다음 내용을 이해하고 검증해야 한다.

```text
데이터 구조
SQL
JOIN
파일 구조
DB 연결
오류 메시지
실행 결과
```

---

# 19. 오늘 발생한 문제와 해결

## 문제 1. DBeaver에서 DB 연결 실패

처음에는 MySQL 연결 과정에서 서버 실행 여부와 포트 설정을 확인했다.

```text
Host: localhost
Port: 3306
```

---

## 문제 2. PHP 페이지 404 오류

```text
404 Not Found
```

원인은 파일 경로나 확장자 문제였다.

프로젝트를 실제 Apache 웹 루트에 배치했다.

```text
C:\xampp\htdocs\course-admin
```

---

## 문제 3. PHP 파일이 TXT 파일로 저장됨

겉으로는 다음처럼 보여도

```text
test_db.php
```

실제로는 다음과 같을 수 있었다.

```text
test_db.php.txt
```

Windows에서 파일 확장자를 표시한 후 수정했다.

```text
보기
→ 표시
→ 파일 이름 확장명
```

---

## 문제 4. db.php를 찾지 못함

다음과 같은 PHP 오류가 발생했다.

```text
Failed opening required
C:\xampp\htdocs\course-admin\config\db.php
```

폴더와 파일 위치를 다시 확인했다.

정상 구조:

```text
course-admin
├── config
│   └── db.php
└── test_db.php
```

수정 후 PHP와 MySQL 연결에 성공했다.

---

# 20. 현재까지 완성한 페이지

현재 다음 5개의 기본 페이지를 만들었다.

```text
index.php
→ Dashboard

students.php
→ 학생 목록 + 학생 등록

instructors.php
→ 강사 목록

courses.php
→ 강좌 목록 + 강사 JOIN

enrollments.php
→ 학생 + 강좌 + 강사 다중 JOIN
```

추가로 DB 연결 확인용 파일도 만들었다.

```text
test_db.php
```

---

# 21. 다음 학습 예정

다음 단계에서는 현재 기능 중심의 페이지에 UI를 적용할 예정이다.

```text
assets/
├── css/
│   └── custom.css
└── js/
    └── app.js
```

추후 진행할 내용:

```text
- Bootstrap 적용
- Sidebar 메뉴
- Dashboard 카드
- 테이블 디자인
- HTML 직접 수정
- CSS 직접 수정
- JavaScript 이벤트
- SQL WHERE / ORDER BY / LIMIT 수정
- GitHub commit / push
```

---

# 오늘의 핵심 정리

오늘 가장 중요하게 배운 것은 다음과 같다.

### 1. 브라우저는 DB에 직접 연결하지 않는다.

```text
Browser
→ PHP
→ MySQL
```

### 2. PHP에서는 PDO를 사용해 MySQL에 연결할 수 있다.

```text
PHP
→ PDO
→ MySQL
```

### 3. SQL 결과를 PHP가 받아 HTML로 출력한다.

```text
SQL
→ DB Result
→ PHP
→ HTML
```

### 4. JOIN을 사용하면 숫자 ID를 실제 이름으로 보여줄 수 있다.

```text
student_id
→ JOIN
→ student_name
```

### 5. 사용자 입력에는 Prepared Statement를 사용한다.

```text
HTML Form
→ POST
→ Prepared Statement
→ INSERT
```

### 6. AI 바이브코딩에서도 DB 구조와 SQL을 이해해야 한다.

AI가 코드를 빠르게 생성해 줄 수 있지만, 어떤 데이터가 어떻게 연결되고 어떤 SQL이 실행되는지는 사람이 이해하고 검증해야 한다.

---

## 한 줄 회고

> 오늘은 SQL을 DBeaver 안에서만 사용하는 단계에서 벗어나, PHP와 MySQL을 연결하여 실제 웹 페이지의 조회와 등록 기능으로 구현해 보았다.
